In [ ]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask

from rfsoc_rfdc.transmitter.single_ch_tx_task import SingleChTxTask
from rfsoc_rfdc.beamformer_task import BeamformerTxTask, BeamformerRxTask

from rfsoc_rfdc.receiver.fmcw_rx_task import FmcwRxTask
from rfsoc_rfdc.receiver.multi_ch_rx_task import MultiChRxTask


from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.mts_task import MtsTask
from rfsoc_rfdc.array_calib_task import ArrayCalibTask
from rfsoc_rfdc.overlay_task import OverlayTask, TASK_STATE

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [ ]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.detection import Detection

In [ ]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v47_8t2r_bf.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 700,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -700
}
ZCU216_CONFIG.update(NEW_CONFIG)

In [ ]:
rfdc_t = RfdcTask(ol, debug_mode=True, board="ZCU216")
mts_t = MtsTask(ol, board="ZCU216", debug_mode=True)

for task in [mts_t, rfdc_t]:
    task.start()
    task.join()

In [ ]:
# calib_t = ArrayCalibTask(ol)
# calib_t.start()
# calib_t.join()

In [ ]:
tx_bf_task = BeamformerTxTask(ol, num_channels=8, debug_mode=True)
tx_bf_task.calib_steer(0)

rx_bf_task = BeamformerRxTask(ol, num_channels=2, debug_mode=True)
rx_bf_task.calib_steer(0)

In [ ]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

# Beam steering experiments
led_t = BlinkLedTask(ol)
led_t.start()

ZCU216_CONFIG['DETECTION_SCHEME'] = Detection(sample_rate=true_samp_rate)

tx_t = SingleChTxTask(ol, waveform_type="FMCW")
tx_t.start()

rx_t = FmcwRxTask(ol, channel_count=2, dp_vect_dim=4)
rx_t.start()

input("Press to stop")

In [ ]:
tx_t.stop()
rx_t.stop()
led_t.stop()